# Natural Language Processing

## Предварительная обработка текстов

### Задача классификации твитов на тональность

In [1]:
# скачаем куски датасета
!wget https://raw.githubusercontent.com/maryszmary/netology_nlp_2021/master/sem1/tweets_sentiment.csv

--2025-09-13 16:37:18--  https://raw.githubusercontent.com/maryszmary/netology_nlp_2021/master/sem1/tweets_sentiment.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 32795904 (31M) [text/plain]
Saving to: ‘tweets_sentiment.csv’

tweets_sentiment.cs 100%[===================>]  31.28M  22.9MB/s    in 1.4s    

2025-09-13 16:37:21 (22.9 MB/s) - ‘tweets_sentiment.csv’ saved [32795904/32795904]



In [9]:
import pandas as pd
import numpy as np
from sklearn.metrics import *
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer

In [4]:
df = pd.read_csv('tweets_sentiment.csv')

In [5]:
df.head()

,text,label
0,мыс на меня обиделась:(\nя ей даже ничего не с...,negative
1,"аааааааааааааааааааа,не хочу на работу :(",negative
2,"У меня какой-то особенный вид ушей! :D, некото...",positive
3,@simonovkon он неплохой человек в жизни. Я ра...,negative
4,"RT @Darina_Lo: Домааааа\nЕхали на такси, пели ...",positive


In [6]:
x_train, x_test, y_train, y_test = train_test_split(df.text, df.label)

In [7]:
print(df.shape)
print(x_train.shape)
print(x_test.shape)

(226834, 2)
(170125,)
(56709,)


### Baseline: Классификация необработанных n-грамм
### Векторизаторы

In [8]:
df['text'].head().tolist()

['мыс на меня обиделась:(\nя ей даже ничего не сделала:(',
 'аааааааааааааааааааа,не хочу на работу :(',
 'У меня какой-то особенный вид ушей! :D, некоторые вакуумные наушники в моих ушах просто не держатся!',
 '@simonovkon  он неплохой человек в жизни. Я работала в шоу-бизе, со многими знакома. Встречаются очень хорошие люди. И не очень(((',
 'RT @Darina_Lo: Домааааа\nЕхали на такси, пели песни, отдыхали.\nКричали на улице:)\nМы настоящяя семья)']

#### CountVectorizer

- Строит для каждого документа (каждой пришедшей ему строки) вектор размерности `n`, где `n` - количество слов или n-грамм во всем корпусе
- Заполняет каждый i-й элемент количеством вхождений слова в данный документ

In [10]:
vec = CountVectorizer(ngram_range=(1, 1))
bow = vec.fit_transform(x_train)

In [11]:
list(vec.vocabulary_.items())[:20]

[('хотя', 234368),
 ('чем', 236700),
 ('это', 241754),
 ('какие', 142718),
 ('внуки', 113450),
 ('большей', 107295),
 ('части', 236386),
 ('дети', 124953),
 ('снаркоманились', 212127),
 ('деградировали', 124082),
 ('вскормленые', 115519),
 ('ворованным', 114506),
 ('не', 165414),
 ('будет', 108523),
 ('них', 168456),
 ('внуков', 113451),
 ('alinkabulova', 11179),
 ('понимаешь', 188747),
 ('меня', 157631),
 ('одна', 172496)]

#### Создание n-грамм с помощью библиотеки nltk

In [19]:
from nltk import ngrams

In [20]:
sent = 'Harry Potter and the Methods of Rationality'.split()
list(ngrams(sent,1)) # униграммы

[('Harry',),
 ('Potter',),
 ('and',),
 ('the',),
 ('Methods',),
 ('of',),
 ('Rationality',)]

In [23]:
sent

['Harry', 'Potter', 'and', 'the', 'Methods', 'of', 'Rationality']

In [21]:
list(ngrams(sent, 2)) # биграммы

[('Harry', 'Potter'),
 ('Potter', 'and'),
 ('and', 'the'),
 ('the', 'Methods'),
 ('Methods', 'of'),
 ('of', 'Rationality')]

In [22]:
list(ngrams(sent, 3)) # триграммы

[('Harry', 'Potter', 'and'),
 ('Potter', 'and', 'the'),
 ('and', 'the', 'Methods'),
 ('the', 'Methods', 'of'),
 ('Methods', 'of', 'Rationality')]

In [25]:
clf = LogisticRegression(random_state=42, solver='liblinear')
clf.fit(bow, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'liblinear'
,max_iter,100
,multi_class,'deprecated'


In [26]:
pred = clf.predict(vec.transform(x_test))
print(classification_report(pred, y_test))

              precision    recall  f1-score   support

    negative       0.77      0.76      0.76     28557
    positive       0.76      0.77      0.76     28152

    accuracy                           0.76     56709
   macro avg       0.76      0.76      0.76     56709
weighted avg       0.76      0.76      0.76     56709

